In [0]:
# ============================================================
# GOLD INCREMENTAL PROCESSING USING SILVER DELTA CDF
# ============================================================
#
# Silver:
#   motordatabricks.utilities_silver.assets
#   motordatabricks.utilities_silver.technicians
#   motordatabricks.utilities_silver.locations
#   motordatabricks.utilities_silver.maintenance
#
# Gold:
#   motordatabricks.utilities_gold.dim_asset
#   motordatabricks.utilities_gold.dim_technician
#   motordatabricks.utilities_gold.dim_location
#   motordatabricks.utilities_gold.dim_date
#   motordatabricks.utilities_gold.dim_maintenance_type
#   motordatabricks.utilities_gold.dim_status
#   motordatabricks.utilities_gold.fact_maintenance
#
# CDF:
#   Only INSERT and UPDATE_POSTIMAGE are processed.
#
# Control table:
#   motordatabricks.utilities_gold.cdf_control
#
# ============================================================


from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. CONFIGURATION
# ============================================================

SILVER_CATALOG = "motordatabricks"
SILVER_SCHEMA = "utilities_silver"

GOLD_CATALOG = "motordatabricks"
GOLD_SCHEMA = "utilities_gold"

CONTROL_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.cdf_control"


SILVER_ASSETS = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.assets"
SILVER_TECHNICIANS = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.technicians"
SILVER_LOCATIONS = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.locations"
SILVER_MAINTENANCE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.maintenance"


GOLD_DIM_ASSET = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_asset"
GOLD_DIM_TECHNICIAN = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_technician"
GOLD_DIM_LOCATION = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_location"
GOLD_DIM_DATE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date"
GOLD_DIM_MAINTENANCE_TYPE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_maintenance_type"
GOLD_DIM_STATUS = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_status"
GOLD_FACT_MAINTENANCE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.fact_maintenance"


# ============================================================
# 2. HELPER - GET CURRENT DELTA VERSION
# ============================================================

def get_current_version(table_name):

    history_df = (
        DeltaTable
        .forName(spark, table_name)
        .history(1)
        .select("version")
    )

    row = history_df.first()

    if row is None:
        raise RuntimeError(
            f"Unable to determine current Delta version for {table_name}"
        )

    return int(row["version"])


# ============================================================
# 3. HELPER - GET LAST PROCESSED VERSION
# ============================================================

def get_last_processed_version(source_table):

    rows = (
        spark.table(CONTROL_TABLE)
        .filter(F.col("source_table") == source_table)
        .select("last_processed_version")
        .limit(1)
        .collect()
    )

    if not rows:
        raise RuntimeError(
            f"No control-table entry found for source table: {source_table}"
        )

    return int(rows[0]["last_processed_version"])


# ============================================================
# 4. HELPER - UPDATE CONTROL TABLE
# ============================================================

def update_control_version(source_table, processed_version):

    spark.sql(f"""
        UPDATE {CONTROL_TABLE}
        SET
            last_processed_version = {processed_version},
            updated_at = current_timestamp()
        WHERE source_table = '{source_table}'
    """)


# ============================================================
# 5. HELPER - READ CDF
# ============================================================

def read_cdf(source_table, start_version, end_version):

    if end_version < start_version:
        return None

    print(
        f"Reading CDF: {source_table} "
        f"from version {start_version} to {end_version}"
    )

    cdf_df = (
        spark.read
        .option("readChangeFeed", "true")
        .option("startingVersion", start_version)
        .option("endingVersion", end_version)
        .table(source_table)
    )

    # We intentionally process only the final image of inserts/updates.
    #
    # update_preimage = old version
    # update_postimage = new version
    #
    # Deletes are NOT processed in this project.
    cdf_df = (
        cdf_df
        .filter(
            F.col("_change_type").isin(
                "insert",
                "update_postimage"
            )
        )
    )

    return cdf_df


# ============================================================
# 6. HELPER - DEDUPLICATE CDF
# ============================================================

def latest_cdf_record(df, business_key):

    if df is None:
        return None

    window_spec = (
        Window
        .partitionBy(business_key)
        .orderBy(
            F.col("_commit_version").desc(),
            F.col("_commit_timestamp").desc()
        )
    )

    return (
        df
        .withColumn("_rn", F.row_number().over(window_spec))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


# ============================================================
# 7. PROCESS ASSETS
# ============================================================

def process_assets():

    source_table = SILVER_ASSETS

    last_version = get_last_processed_version(source_table)
    current_version = get_current_version(source_table)

    print(
        f"\nASSETS | "
        f"Last processed = {last_version}, "
        f"Current = {current_version}"
    )

    if current_version <= last_version:

        print("ASSETS: No new changes.")

        return

    cdf_df = read_cdf(
        source_table,
        last_version + 1,
        current_version
    )

    cdf_df = latest_cdf_record(
        cdf_df,
        "asset_id"
    )

    if cdf_df.isEmpty():

        print("ASSETS: No insert/update changes found.")

        update_control_version(
            source_table,
            current_version
        )

        return

    source_df = (
        cdf_df
        .select(
            "asset_id",
            "asset_type",
            "manufacturer",
            "model",
            "serial_number",
            F.to_date("installation_date").alias("installation_date"),
            "location_id",
            "city",
            "region",
            "criticality",
            "asset_status"
        )
        .withColumn(
            "asset_key",
            F.xxhash64("asset_id")
        )
    )

    target = DeltaTable.forName(
        spark,
        GOLD_DIM_ASSET
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.asset_id = s.asset_id"
        )
        .whenMatchedUpdate(
            set={
                "asset_type": "s.asset_type",
                "manufacturer": "s.manufacturer",
                "model": "s.model",
                "serial_number": "s.serial_number",
                "installation_date": "s.installation_date",
                "location_id": "s.location_id",
                "city": "s.city",
                "region": "s.region",
                "criticality": "s.criticality",
                "asset_status": "s.asset_status"
            }
        )
        .whenNotMatchedInsert(
            values={
                "asset_key": "s.asset_key",
                "asset_id": "s.asset_id",
                "asset_type": "s.asset_type",
                "manufacturer": "s.manufacturer",
                "model": "s.model",
                "serial_number": "s.serial_number",
                "installation_date": "s.installation_date",
                "location_id": "s.location_id",
                "city": "s.city",
                "region": "s.region",
                "criticality": "s.criticality",
                "asset_status": "s.asset_status"
            }
        )
        .execute()
    )

    # Update checkpoint ONLY after MERGE succeeds.
    update_control_version(
        source_table,
        current_version
    )

    print(
        f"ASSETS: Successfully processed through version "
        f"{current_version}"
    )


# ============================================================
# 8. PROCESS TECHNICIANS
# ============================================================

def process_technicians():

    source_table = SILVER_TECHNICIANS

    last_version = get_last_processed_version(source_table)
    current_version = get_current_version(source_table)

    print(
        f"\nTECHNICIANS | "
        f"Last processed = {last_version}, "
        f"Current = {current_version}"
    )

    if current_version <= last_version:

        print("TECHNICIANS: No new changes.")

        return

    cdf_df = read_cdf(
        source_table,
        last_version + 1,
        current_version
    )

    cdf_df = latest_cdf_record(
        cdf_df,
        "technician_id"
    )

    if cdf_df.isEmpty():

        print("TECHNICIANS: No insert/update changes found.")

        update_control_version(
            source_table,
            current_version
        )

        return

    source_df = (
        cdf_df
        .select(
            "technician_id",
            "technician_name",
            "skill_level",
            "team",
            "experience_years"
        )
        .withColumn(
            "technician_key",
            F.xxhash64("technician_id")
        )
    )

    target = DeltaTable.forName(
        spark,
        GOLD_DIM_TECHNICIAN
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.technician_id = s.technician_id"
        )
        .whenMatchedUpdate(
            set={
                "technician_name": "s.technician_name",
                "skill_level": "s.skill_level",
                "team": "s.team",
                "experience_years": "s.experience_years"
            }
        )
        .whenNotMatchedInsert(
            values={
                "technician_key": "s.technician_key",
                "technician_id": "s.technician_id",
                "technician_name": "s.technician_name",
                "skill_level": "s.skill_level",
                "team": "s.team",
                "experience_years": "s.experience_years"
            }
        )
        .execute()
    )

    update_control_version(
        source_table,
        current_version
    )

    print(
        f"TECHNICIANS: Successfully processed through version "
        f"{current_version}"
    )


# ============================================================
# 9. PROCESS LOCATIONS
# ============================================================

def process_locations():

    source_table = SILVER_LOCATIONS

    last_version = get_last_processed_version(source_table)
    current_version = get_current_version(source_table)

    print(
        f"\nLOCATIONS | "
        f"Last processed = {last_version}, "
        f"Current = {current_version}"
    )

    if current_version <= last_version:

        print("LOCATIONS: No new changes.")

        return

    cdf_df = read_cdf(
        source_table,
        last_version + 1,
        current_version
    )

    cdf_df = latest_cdf_record(
        cdf_df,
        "location_id"
    )

    if cdf_df.isEmpty():

        print("LOCATIONS: No insert/update changes found.")

        update_control_version(
            source_table,
            current_version
        )

        return

    source_df = (
        cdf_df
        .select(
            "location_id",
            "city",
            "region"
        )
        .withColumn(
            "location_key",
            F.xxhash64("location_id")
        )
    )

    target = DeltaTable.forName(
        spark,
        GOLD_DIM_LOCATION
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.location_id = s.location_id"
        )
        .whenMatchedUpdate(
            set={
                "city": "s.city",
                "region": "s.region"
            }
        )
        .whenNotMatchedInsert(
            values={
                "location_key": "s.location_key",
                "location_id": "s.location_id",
                "city": "s.city",
                "region": "s.region"
            }
        )
        .execute()
    )

    update_control_version(
        source_table,
        current_version
    )

    print(
        f"LOCATIONS: Successfully processed through version "
        f"{current_version}"
    )


# ============================================================
# 10. PROCESS MAINTENANCE TYPE DIMENSION
# ============================================================

def process_maintenance_type(maintenance_cdf):

    if maintenance_cdf is None:
        return

    if maintenance_cdf.isEmpty():
        return

    source_df = (
        maintenance_cdf
        .select(
            "maintenance_type",
            "maintenance_category",
            "planned_flag",
            "failure_type",
            "failure_severity",
            "priority"
        )
        .dropDuplicates(
            [
                "maintenance_type",
                "maintenance_category",
                "planned_flag",
                "failure_type",
                "failure_severity",
                "priority"
            ]
        )
        .withColumn(
            "maintenance_type_key",
            F.xxhash64(
                "maintenance_type",
                "maintenance_category",
                "planned_flag",
                "failure_type",
                "failure_severity",
                "priority"
            )
        )
    )

    target = DeltaTable.forName(
        spark,
        GOLD_DIM_MAINTENANCE_TYPE
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            """
            t.maintenance_type_key =
            s.maintenance_type_key
            """
        )
        .whenNotMatchedInsert(
            values={
                "maintenance_type_key":
                    "s.maintenance_type_key",
                "maintenance_type":
                    "s.maintenance_type",
                "maintenance_category":
                    "s.maintenance_category",
                "planned_flag":
                    "s.planned_flag",
                "failure_type":
                    "s.failure_type",
                "failure_severity":
                    "s.failure_severity",
                "priority":
                    "s.priority"
            }
        )
        .execute()
    )


# ============================================================
# 11. PROCESS STATUS DIMENSION
# ============================================================

def process_status(maintenance_cdf):

    if maintenance_cdf is None:
        return

    if maintenance_cdf.isEmpty():
        return

    source_df = (
        maintenance_cdf
        .select(
            "work_order_status",
            "resolution_code"
        )
        .dropDuplicates(
            [
                "work_order_status",
                "resolution_code"
            ]
        )
        .withColumn(
            "status_key",
            F.xxhash64(
                "work_order_status",
                "resolution_code"
            )
        )
    )

    target = DeltaTable.forName(
        spark,
        GOLD_DIM_STATUS
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.status_key = s.status_key"
        )
        .whenNotMatchedInsert(
            values={
                "status_key":
                    "s.status_key",
                "work_order_status":
                    "s.work_order_status",
                "resolution_code":
                    "s.resolution_code"
            }
        )
        .execute()
    )


# ============================================================
# 12. PROCESS DATE DIMENSION
# ============================================================

def process_date_dimension(maintenance_cdf):

    if maintenance_cdf is None:
        return

    if maintenance_cdf.isEmpty():
        return

    dates_df = (
        maintenance_cdf
        .select(
            F.to_date("created_date").alias("date"),
            F.to_date("scheduled_date").alias("scheduled_date"),
            F.to_date("completed_date").alias("completed_date")
        )
    )

    # Collect all possible dates from the three date columns.
    dates_df = (
        dates_df
        .select(
            F.explode(
                F.array(
                    "date",
                    "scheduled_date",
                    "completed_date"
                )
            ).alias("date")
        )
        .filter(F.col("date").isNotNull())
        .dropDuplicates(["date"])
    )

    source_df = (
        dates_df
        .withColumn(
            "date_key",
            F.date_format("date", "yyyyMMdd").cast("int")
        )
        .withColumn(
            "year",
            F.year("date")
        )
        .withColumn(
            "quarter",
            F.quarter("date")
        )
        .withColumn(
            "month",
            F.month("date")
        )
        .withColumn(
            "month_name",
            F.date_format("date", "MMMM")
        )
        .withColumn(
            "week",
            F.weekofyear("date")
        )
        .withColumn(
            "day",
            F.dayofmonth("date")
        )
        .withColumn(
            "day_of_week",
            F.dayofweek("date")
        )
        .withColumn(
            "day_name",
            F.date_format("date", "EEEE")
        )
        .withColumn(
            "is_weekend",
            F.dayofweek("date").isin([1, 7])
        )
    )

    target = DeltaTable.forName(
        spark,
        GOLD_DIM_DATE
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.date_key = s.date_key"
        )
        .whenNotMatchedInsert(
            values={
                "date_key": "s.date_key",
                "date": "s.date",
                "year": "s.year",
                "quarter": "s.quarter",
                "month": "s.month",
                "month_name": "s.month_name",
                "week": "s.week",
                "day": "s.day",
                "day_of_week": "s.day_of_week",
                "day_name": "s.day_name",
                "is_weekend": "s.is_weekend"
            }
        )
        .execute()
    )


# ============================================================
# 13. PROCESS FACT MAINTENANCE
# ============================================================

def process_fact_maintenance(maintenance_cdf):

    if maintenance_cdf is None:
        return

    if maintenance_cdf.isEmpty():
        return

    # --------------------------------------------------------
    # Load required Gold dimensions
    # --------------------------------------------------------

    dim_asset_df = (
        spark.table(GOLD_DIM_ASSET)
        .select(
            "asset_id",
            "asset_key"
        )
    )

    dim_technician_df = (
        spark.table(GOLD_DIM_TECHNICIAN)
        .select(
            "technician_id",
            "technician_key"
        )
    )

    dim_location_df = (
        spark.table(GOLD_DIM_LOCATION)
        .select(
            "location_id",
            "location_key"
        )
    )

    dim_maintenance_type_df = (
        spark.table(GOLD_DIM_MAINTENANCE_TYPE)
        .select(
            "maintenance_type_key",
            "maintenance_type",
            "maintenance_category",
            "planned_flag",
            "failure_type",
            "failure_severity",
            "priority"
        )
    )

    dim_status_df = (
        spark.table(GOLD_DIM_STATUS)
        .select(
            "status_key",
            "work_order_status",
            "resolution_code"
        )
    )


    # --------------------------------------------------------
    # Build fact source
    # --------------------------------------------------------

    source_df = (
        maintenance_cdf.alias("m")

        .join(
            dim_asset_df.alias("a"),
            F.col("m.asset_id") == F.col("a.asset_id"),
            "left"
        )

        .join(
            dim_technician_df.alias("t"),
            F.col("m.technician_id") ==
            F.col("t.technician_id"),
            "left"
        )

        .join(
            dim_location_df.alias("l"),
            F.col("m.location_id") ==
            F.col("l.location_id"),
            "left"
        )

        .join(
            dim_maintenance_type_df.alias("mt"),
            (
                (F.col("m.maintenance_type") ==
                 F.col("mt.maintenance_type"))
                &
                (F.col("m.maintenance_category") ==
                 F.col("mt.maintenance_category"))
                &
                (F.col("m.planned_flag") ==
                 F.col("mt.planned_flag"))
                &
                (F.col("m.failure_type") ==
                 F.col("mt.failure_type"))
                &
                (F.col("m.failure_severity") ==
                 F.col("mt.failure_severity"))
                &
                (F.col("m.priority") ==
                 F.col("mt.priority"))
            ),
            "left"
        )

        .join(
            dim_status_df.alias("st"),
            F.xxhash64(
                F.col("m.work_order_status"),
                F.col("m.resolution_code")
            ) == F.col("st.status_key"),
            "left"
        )

        .select(
            F.xxhash64(
                F.col("m.work_order_id")
            ).alias("maintenance_key"),

            F.col("m.work_order_id"),

            F.col("a.asset_key"),

            F.col("t.technician_key"),

            F.col("l.location_key"),

            F.col("mt.maintenance_type_key"),

            F.col("st.status_key"),

            F.when(
                F.col("m.created_date").isNotNull(),
                F.date_format(
                    F.to_date("m.created_date"),
                    "yyyyMMdd"
                ).cast("int")
            ).alias("created_date_key"),

            F.when(
                F.col("m.scheduled_date").isNotNull(),
                F.date_format(
                    F.to_date("m.scheduled_date"),
                    "yyyyMMdd"
                ).cast("int")
            ).alias("scheduled_date_key"),

            F.when(
                F.col("m.completed_date").isNotNull(),
                F.date_format(
                    F.to_date("m.completed_date"),
                    "yyyyMMdd"
                ).cast("int")
            ).alias("completed_date_key"),

            F.col("m.downtime_hours"),

            F.col("m.labor_hours"),

            F.col("m.parts_used"),

            F.col("m.parts_cost"),

            F.col("m.labor_cost"),

            F.col("m.total_cost"),

            F.col("m.created_date"),

            F.col("m.scheduled_date"),

            F.col("m.completed_date"),

            F.col("m.source_system"),

            F.col("m.updated_at")
        )
    )


    # --------------------------------------------------------
    # Validate dimension lookups
    # --------------------------------------------------------
    #
    # If a source record references an ID that doesn't exist
    # in its corresponding dimension, don't silently create
    # an invalid fact row.
    #
    # The dimensions should already have been processed above.
    # --------------------------------------------------------

    invalid_fk_df = (
        source_df
        .filter(
            F.col("asset_key").isNull()
            |
            F.col("technician_key").isNull()
            |
            F.col("location_key").isNull()
            |
            F.col("maintenance_type_key").isNull()
            |
            F.col("status_key").isNull()
        )
        .select("work_order_id")
    )

    invalid_count = invalid_fk_df.limit(1).count()

    if invalid_count > 0:

        invalid_ids = [
            row["work_order_id"]
            for row in invalid_fk_df
            .limit(20)
            .collect()
        ]

        raise RuntimeError(
            "Fact processing stopped because one or more "
            "dimension lookups failed. "
            f"Example work_order_id values: {invalid_ids}"
        )


    # --------------------------------------------------------
    # MERGE into fact
    # --------------------------------------------------------

    target = DeltaTable.forName(
        spark,
        GOLD_FACT_MAINTENANCE
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.work_order_id = s.work_order_id"
        )

        .whenMatchedUpdate(
            set={

                "asset_key":
                    "s.asset_key",

                "technician_key":
                    "s.technician_key",

                "location_key":
                    "s.location_key",

                "maintenance_type_key":
                    "s.maintenance_type_key",

                "status_key":
                    "s.status_key",

                "created_date_key":
                    "s.created_date_key",

                "scheduled_date_key":
                    "s.scheduled_date_key",

                "completed_date_key":
                    "s.completed_date_key",

                "downtime_hours":
                    "s.downtime_hours",

                "labor_hours":
                    "s.labor_hours",

                "parts_used":
                    "s.parts_used",

                "parts_cost":
                    "s.parts_cost",

                "labor_cost":
                    "s.labor_cost",

                "total_cost":
                    "s.total_cost",

                "created_date":
                    "s.created_date",

                "scheduled_date":
                    "s.scheduled_date",

                "completed_date":
                    "s.completed_date",

                "source_system":
                    "s.source_system",

                "updated_at":
                    "s.updated_at"
            }
        )

        .whenNotMatchedInsert(
            values={

                "maintenance_key":
                    "s.maintenance_key",

                "work_order_id":
                    "s.work_order_id",

                "asset_key":
                    "s.asset_key",

                "technician_key":
                    "s.technician_key",

                "location_key":
                    "s.location_key",

                "maintenance_type_key":
                    "s.maintenance_type_key",

                "status_key":
                    "s.status_key",

                "created_date_key":
                    "s.created_date_key",

                "scheduled_date_key":
                    "s.scheduled_date_key",

                "completed_date_key":
                    "s.completed_date_key",

                "downtime_hours":
                    "s.downtime_hours",

                "labor_hours":
                    "s.labor_hours",

                "parts_used":
                    "s.parts_used",

                "parts_cost":
                    "s.parts_cost",

                "labor_cost":
                    "s.labor_cost",

                "total_cost":
                    "s.total_cost",

                "created_date":
                    "s.created_date",

                "scheduled_date":
                    "s.scheduled_date",

                "completed_date":
                    "s.completed_date",

                "source_system":
                    "s.source_system",

                "updated_at":
                    "s.updated_at"
            }
        )

        .execute()
    )


# ============================================================
# 14. MAIN PROCESS
# ============================================================

print("=" * 70)
print("STARTING GOLD CDF INCREMENTAL PROCESSING")
print("=" * 70)


# ------------------------------------------------------------
# STEP 1
# Process independent dimensions first
# ------------------------------------------------------------

process_assets()

process_technicians()

process_locations()


# ------------------------------------------------------------
# STEP 2
# Read Maintenance CDF ONCE
# ------------------------------------------------------------
#
# We need the maintenance changes for:
#
#   dim_maintenance_type
#   dim_status
#   dim_date
#   fact_maintenance
#
# We therefore determine its version once and reuse it.
# ------------------------------------------------------------

maintenance_last_version = get_last_processed_version(
    SILVER_MAINTENANCE
)

maintenance_current_version = get_current_version(
    SILVER_MAINTENANCE
)


print(
    f"\nMAINTENANCE | "
    f"Last processed = {maintenance_last_version}, "
    f"Current = {maintenance_current_version}"
)


if maintenance_current_version <= maintenance_last_version:

    print("MAINTENANCE: No new changes.")

else:

    maintenance_cdf = read_cdf(
        SILVER_MAINTENANCE,
        maintenance_last_version + 1,
        maintenance_current_version
    )

    maintenance_cdf = latest_cdf_record(
        maintenance_cdf,
        "work_order_id"
    )


    # --------------------------------------------------------
    # Process dependent dimensions BEFORE fact
    # --------------------------------------------------------

    process_maintenance_type(
        maintenance_cdf
    )

    process_status(
        maintenance_cdf
    )

    process_date_dimension(
        maintenance_cdf
    )


    # --------------------------------------------------------
    # Process fact
    # --------------------------------------------------------

    process_fact_maintenance(
        maintenance_cdf
    )


    # --------------------------------------------------------
    # Update maintenance checkpoint ONLY after
    # all maintenance-related Gold processing succeeds.
    # --------------------------------------------------------

    update_control_version(
        SILVER_MAINTENANCE,
        maintenance_current_version
    )

    print(
        f"MAINTENANCE: Successfully processed through version "
        f"{maintenance_current_version}"
    )


# ============================================================
# 15. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL CDF CONTROL TABLE")
print("=" * 70)

display(
    spark.table(CONTROL_TABLE)
    .orderBy("source_table")
)


print("\n" + "=" * 70)
print("GOLD ROW COUNTS")
print("=" * 70)

gold_tables = [
    GOLD_DIM_ASSET,
    GOLD_DIM_TECHNICIAN,
    GOLD_DIM_LOCATION,
    GOLD_DIM_DATE,
    GOLD_DIM_MAINTENANCE_TYPE,
    GOLD_DIM_STATUS,
    GOLD_FACT_MAINTENANCE
]

for table_name in gold_tables:

    count = spark.table(table_name).count()

    print(
        f"{table_name}: {count:,} rows"
    )


print("\n" + "=" * 70)
print("GOLD CDF INCREMENTAL PROCESSING COMPLETED")
print("=" * 70)

STARTING GOLD CDF INCREMENTAL PROCESSING

ASSETS | Last processed = 2, Current = 2
ASSETS: No new changes.

TECHNICIANS | Last processed = 2, Current = 2
TECHNICIANS: No new changes.

LOCATIONS | Last processed = 2, Current = 2
LOCATIONS: No new changes.

MAINTENANCE | Last processed = 5, Current = 5
MAINTENANCE: No new changes.

FINAL CDF CONTROL TABLE


source_table,last_processed_version,updated_at
motordatabricks.utilities_silver.assets,2,2026-09-03T01:07:03.372Z
motordatabricks.utilities_silver.locations,2,2026-09-03T01:07:03.372Z
motordatabricks.utilities_silver.maintenance,5,2026-09-03T01:07:03.372Z
motordatabricks.utilities_silver.technicians,2,2026-09-03T01:07:03.372Z



GOLD ROW COUNTS
motordatabricks.utilities_gold.dim_asset: 500 rows
motordatabricks.utilities_gold.dim_technician: 100 rows
motordatabricks.utilities_gold.dim_location: 72 rows
motordatabricks.utilities_gold.dim_date: 188 rows
motordatabricks.utilities_gold.dim_maintenance_type: 660 rows
motordatabricks.utilities_gold.dim_status: 11 rows
motordatabricks.utilities_gold.fact_maintenance: 3,600 rows

GOLD CDF INCREMENTAL PROCESSING COMPLETED
